In [4]:
import zipfile
from multiprocessing import cpu_count
import splitfolders
from tensorflow import keras
import numpy as np
from tqdm import tqdm
import shutil
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import regularizers
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model


In [2]:
print("TensorFlow version:", tf.__version__)
print("CPU cores:", tf.config.threading.get_intra_op_parallelism_threads())
print(f"CPU cores detected: {cpu_count()}")

TensorFlow version: 2.20.0
CPU cores: 0
CPU cores detected: 12


In [5]:
zip_path = "bovine_breeds.zip"


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [7]:
data_dir = 'BovineBreeds_Augmented'
for cls in os.listdir(data_dir):
    path = os.path.join(data_dir, cls)
    print(f"{cls}: {len(os.listdir(path))} images")

.ipynb_checkpoints: 0 images
Alambadi: 246 images
Amritmahal: 251 images
Ayrshire: 234 images
Banni: 235 images
Bargur: 249 images
Bhadawari: 257 images
Brown_Swiss: 225 images
Dangi: 262 images
Deoni: 246 images
Gir: 244 images
Holstein_Friesian: 244 images
Jaffrabadi: 240 images
Jersey: 253 images
Kangayam: 254 images
Kasargod: 250 images
Kenkatha: 288 images
Kherigarh: 306 images
Krishna_Valley: 186 images
Mehsana: 249 images
Murrah: 223 images
Nagori: 255 images
Nili_Ravi: 255 images
Nimari: 261 images
Ongole: 240 images
Red_Sindhi: 246 images
Sahiwal: 244 images
Surti: 272 images
Tharparkar: 267 images
Umblachery: 269 images


In [8]:
input_folder = "BovineBreeds_Augmented"
output_folder = "split_data"

splitfolders.ratio(input_folder,
                   output=output_folder,
                   seed=42,
                   ratio=(0.7, 0.2, 0.1))

Copying files: 7250 files [00:39, 183.44 files/s]


In [13]:
def remove_class(path):
    class_to_remove = '.ipynb_checkpoints'
    folder_path = os.path.join(path, class_to_remove)

    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"{class_to_remove} removed successfully")
    else:
        print("Class not found")

remove_class('split_data/train')
remove_class('split_data/val')
remove_class('split_data/test')

.ipynb_checkpoints removed successfully
.ipynb_checkpoints removed successfully
.ipynb_checkpoints removed successfully


In [14]:
train_dir = 'split_data/train'

class_names_from_dir = sorted(os.listdir(train_dir))
print(class_names_from_dir)

['Alambadi', 'Amritmahal', 'Ayrshire', 'Banni', 'Bargur', 'Bhadawari', 'Brown_Swiss', 'Dangi', 'Deoni', 'Gir', 'Holstein_Friesian', 'Jaffrabadi', 'Jersey', 'Kangayam', 'Kasargod', 'Kenkatha', 'Kherigarh', 'Krishna_Valley', 'Mehsana', 'Murrah', 'Nagori', 'Nili_Ravi', 'Nimari', 'Ongole', 'Red_Sindhi', 'Sahiwal', 'Surti', 'Tharparkar', 'Umblachery']


In [6]:
IMG_SIZE = (300, 300)


train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,  # ✅ EfficientNet-specific
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
)


val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
)

train_generator = train_datagen.flow_from_directory(
    'split_data/train',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'
)


val_generator = val_datagen.flow_from_directory(
    'split_data/val',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical',

)



test_generator = val_datagen.flow_from_directory(
    'split_data/test',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'

)

Found 5062 images belonging to 30 classes.
Found 1439 images belonging to 30 classes.
Found 749 images belonging to 30 classes.


In [7]:
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300, 300, 3))

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x1 = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
x2 = layers.Dropout(0.2)(x1)
x3 = Dense(64, activation = 'relu', kernel_regularizer= regularizers.l2(0.001))(x2)
x4 = layers.Dropout(0.2)(x3)
x5 = Dense(30, activation='softmax')(x4)
model = Model(inputs=base_model.input, outputs=x5)


model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 300, 300,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 300, 300,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 300, 300,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 301, 301,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 150, 150,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 150, 150,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 150, 150,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 150, 150,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 150, 150,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 150, 150,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 150, 150,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 150, 150,  │        960 │ block1a_se_excit

 Total params: 10,990,477 (41.93 MB)

 Trainable params: 206,942 (808.37 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

In [8]:
# Custom callback to monitor CPU during training
import tensorflow.keras.callbacks as callbacks




checkpoint = ModelCheckpoint(
    filepath="/content/drive/MyDrive/Breed Classification/model/best_model.keras",
    monitor="val_accuracy",        # metric to monitor
    mode="max",                    # because we want maximum accuracy
    save_best_only=True,           # saves only when model improves
    verbose=1
)

early_stop = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)



history = model.fit(train_generator, validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, checkpoint],     
    verbose = 1

)

Epoch 1/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3148 - loss: 2.8973
Epoch 1: val_accuracy improved from None to 0.68033, saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras
159/159 ━━━━━━━━━━━━━━━━━━━━ 401s 2s/step - accuracy: 0.4631 - loss: 2.3413 - val_accuracy: 0.6803 - val_loss: 1.4654
Epoch 2/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6506 - loss: 1.5528
Epoch 2: val_accuracy improved from 0.68033 to 0.73384, saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Breed Classification/model/best_model.keras
159/159 ━━━━━━━━━━━━━━━━━━━━ 407s 3s/step - accuracy: 0.6578 - loss: 1.5026 - val_accuracy: 0.7338 - val_loss: 1.2166
Epoch 3/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6943 - loss: 1.3161
Epoch 3: val_accuracy improved

In [9]:
model_path = "best_model.keras"

loaded_model = load_model(model_path)

loss, accuracy = loaded_model.evaluate(test_generator)
print(loss)
print(accuracy)

24/24 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.8344 - loss: 0.9538
0.9538195133209229
0.8344459533691406


In [11]:
from sklearn.metrics import classification_report

pred_probs = model.predict(test_generator)

y_pred = np.argmax(pred_probs, axis=1)


y_true = test_generator.classes

labels = np.unique(np.concatenate([y_true, y_pred]))
class_names = list(test_generator.class_indices.keys())
target_names = [class_names[i] for i in labels]

report = classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=target_names
)

print(report)

24/24 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step
                   precision    recall  f1-score   support

         Alambadi       0.00      0.00      0.00        25
       Amritmahal       0.04      0.04      0.04        26
         Ayrshire       0.00      0.00      0.00        25
            Banni       0.03      0.04      0.04        24
           Bargur       0.04      0.04      0.04        26
        Bhadawari       0.07      0.07      0.07        27
      Brown_Swiss       0.10      0.09      0.09        23
            Dangi       0.04      0.04      0.04        27
            Deoni       0.00      0.00      0.00        25
              Gir       0.00      0.00      0.00        26
Holstein_Friesian       0.03      0.04      0.04        26
       Jaffrabadi       0.12      0.12      0.12        24
           Jersey       0.11      0.12      0.11        26
         Kangayam       0.03      0.04      0.03        27
         Kasargod       0.00      0.00      0.00        25
         Kenkath

In [12]:
y_true = test_generator.classes
y_pred = np.argmax(model.predict(test_generator), axis=1)

labels = sorted(set(y_true) | set(y_pred))

class_names = list(test_generator.class_indices.keys())

print(classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=[class_names[i] for i in labels]
))

24/24 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step
                   precision    recall  f1-score   support

         Alambadi       0.00      0.00      0.00        25
       Amritmahal       0.09      0.08      0.08        26
         Ayrshire       0.04      0.04      0.04        25
            Banni       0.03      0.04      0.04        24
           Bargur       0.00      0.00      0.00        26
        Bhadawari       0.03      0.04      0.04        27
      Brown_Swiss       0.05      0.04      0.05        23
            Dangi       0.04      0.04      0.04        27
            Deoni       0.05      0.04      0.04        25
              Gir       0.00      0.00      0.00        26
Holstein_Friesian       0.00      0.00      0.00        26
       Jaffrabadi       0.00      0.00      0.00        24
           Jersey       0.00      0.00      0.00        26
         Kangayam       0.10      0.11      0.10        27
         Kasargod       0.00      0.00      0.00        25
         Kenkath